# Описание

Код для создания тренировочной и тестовой выборки из основного и дополнительного набора признаков на основании отбора наиболее важных признаков.

В файле kyberpolka_dataset_explore.ipynb проведен предварительный анализ важности признаков на объединенной подвыборке в 5% строк из основных и дополнительных признаков. На  подвыборке обучены RandomForestClassifier и CatboostClassifier и рассчитаны усредненные по всем целевым переменным важности признаков. Отсортированные по важности наборы сохранены в файлах feature_importance_rf.json и feature_importance_cat.json.

Для объединения двух наборов рассчитывается средневзвешенная сумма важностей rf и catboost с весами 0.4 и 0.6 соответственно. 0.6/0.4 — это компромисс, дающий чуть больше веса более точному методу: CatBoost обычно показывает лучшее качество на табличных данных с категориальными признаками. 

Из отсортированного по комбинированной важности списка берётся нужное количество признаков, дающих 90% или 95% важности. Затем эти признаки разделяются на два списка: те, что присутствуют в основном датасете (top_main_features), и те, что в дополнительном (top_extra_features).

Для набора дополнительных признаков используется "ленивая загрузка" с целью экономии памяти. С той же целью производится приведение к Float32 и Int32 вместо Float64 и Int64.

Затем основной и дополнительный наборы объединяются по customer_id (left join) в единую матрицу признаков для обучения. 

Тестовая выборка обрабатывается аналогично.

In [1]:
import gc
import numpy as np
import polars as pl
import json
import pandas as pd

In [2]:
!pip install polars

In [3]:
try:
    import pyarrow
    print("PyArrow is installed. Version:", pyarrow.__version__)
except ModuleNotFoundError:
    print("PyArrow is NOT found in this environment. You installed it in a different one.")

PyArrow is installed. Version: 23.0.1


In [4]:
train = pl.read_parquet('/kaggle/input/datasets/mashamalyshkina/kyberpolka-dataset/train_main_features.parquet')
test = pl.read_parquet('/kaggle/input/datasets/mashamalyshkina/kyberpolka-dataset/test_main_features.parquet')

In [5]:
test_extra = pl.read_parquet('/kaggle/input/datasets/mashamalyshkina/kyberpolka-dataset/test_extra_features.parquet')

In [6]:
feature_importance_rf = pd.read_json('/kaggle/input/datasets/mashamalyshkina/kyberpolka-dataset/feature_importance_rf.json', orient='records')
print("Загружено из JSON:")
print(feature_importance_rf.head())
feature_importance_cat = pd.read_json('/kaggle/input/datasets/mashamalyshkina/kyberpolka-dataset/feature_importance_cat.json', orient='records')
print("Загружено из JSON:")
print(feature_importance_cat.head())

Загружено из JSON:
            feature  importance_rf
0    num_feature_87       0.004691
1    num_feature_62       0.003942
2    num_feature_27       0.003800
3   num_feature_879       0.003702
4  num_feature_1377       0.003294
Загружено из JSON:
            feature  importance_cat
0    num_feature_41        3.462705
1    num_feature_22        1.337958
2  num_feature_1309        1.239737
3  num_feature_1644        1.128667
4    num_feature_42        1.092923


In [7]:
# объединение по имени признака
combined = feature_importance_cat.merge(
    feature_importance_rf, 
    on='feature', 
    suffixes=('_cat', '_rf')
)

# нормализовация
combined['imp_cat_norm'] = combined['importance_cat'] / combined['importance_cat'].max()
combined['imp_rf_norm'] = combined['importance_rf'] / combined['importance_rf'].max()

# комбинированная важность
combined['importance_combined'] = 0.6 * combined['imp_cat_norm'] + 0.4 * combined['imp_rf_norm']

# сортировка и кумулята
combined = combined.sort_values('importance_combined', ascending=False)
combined['cumulative_pct'] = combined['importance_combined'].cumsum() / combined['importance_combined'].sum()

# отбор по порогам
n_90 = (combined['cumulative_pct'] <= 0.9).sum() + 1
n_95 = (combined['cumulative_pct'] <= 0.95).sum() + 1

top_90_features = combined.head(n_90)['feature'].tolist()
top_95_features = combined.head(n_95)['feature'].tolist()

In [8]:
top_main_features = [f for f in top_95_features if f in train.columns]
top_extra_features = [f for f in top_95_features if f in test_extra.columns]

print(f"\nВыбрано признаков: {len(top_95_features)}")
print(f"  Из основного набора: {len(top_main_features)}")
print(f"  Из дополнительного: {len(top_extra_features)}")


Выбрано признаков: 1623
  Из основного набора: 149
  Из дополнительного: 1474


In [9]:
train_main = train[['customer_id'] + top_main_features]
train_main.head()

customer_id,num_feature_41,num_feature_87,num_feature_27,num_feature_117,num_feature_62,num_feature_76,num_feature_42,num_feature_71,num_feature_25,num_feature_88,num_feature_22,num_feature_23,num_feature_126,num_feature_35,num_feature_85,num_feature_36,num_feature_116,num_feature_68,num_feature_73,num_feature_46,num_feature_24,num_feature_61,cat_feature_8,num_feature_95,cat_feature_66,num_feature_58,num_feature_60,num_feature_26,num_feature_16,num_feature_69,num_feature_15,num_feature_52,num_feature_130,num_feature_6,num_feature_63,num_feature_104,…,cat_feature_15,num_feature_86,num_feature_45,cat_feature_37,cat_feature_35,cat_feature_23,num_feature_65,cat_feature_42,num_feature_103,cat_feature_57,cat_feature_14,cat_feature_55,num_feature_118,num_feature_2,cat_feature_5,num_feature_74,cat_feature_33,num_feature_112,cat_feature_25,cat_feature_53,cat_feature_31,cat_feature_21,cat_feature_18,num_feature_93,cat_feature_65,num_feature_92,num_feature_78,cat_feature_4,cat_feature_48,cat_feature_19,cat_feature_54,num_feature_44,cat_feature_51,num_feature_37,cat_feature_34,cat_feature_30,cat_feature_61
i32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1000001,0.445712,null,-0.020738,-0.019079,-0.02543,-0.027542,-0.002153,null,null,null,null,null,-0.445279,0.453232,null,-0.497578,-0.493959,-0.363338,-0.341678,-0.008472,-0.542784,null,2.0,-0.143428,2.0,-0.054061,-0.004373,null,null,null,-0.211947,null,-0.418616,-0.307721,null,0.450342,…,2.0,-0.054053,null,2.0,0.0,2.0,-0.290711,2.0,-0.061488,2.0,1.0,2.0,null,null,2.0,null,0.0,null,0.0,2.0,2.0,0.0,2.0,-0.00859,0.0,-0.046292,-0.097437,1.0,0.0,2.0,2.0,null,2.0,-0.040013,212.0,2.0,2.0
1000002,-0.583389,-0.018017,-0.004672,-0.014154,-0.023054,-0.027717,-0.001728,null,null,null,null,null,1.550722,0.346346,2.330621,0.716116,-0.256445,1.179635,-0.341678,-0.013828,-0.571248,0.449687,0.0,-0.163148,0.0,-0.044567,-0.007853,-0.139725,-0.005735,null,0.374087,-0.066879,-0.805771,-0.307721,null,1.998252,…,1.0,-0.054053,null,0.0,1.0,0.0,-0.290711,0.0,-0.061488,0.0,0.0,1.0,null,-0.210019,0.0,null,0.0,null,0.0,0.0,0.0,0.0,0.0,-0.00859,0.0,-0.046292,-0.097437,1.0,0.0,0.0,0.0,null,0.0,-0.040013,212.0,0.0,0.0
1000003,-0.197476,null,-0.020788,-0.019124,-0.025546,-0.02765,null,null,null,null,null,null,-0.475778,0.427056,null,-0.983056,-0.57313,-0.591927,-0.341678,-0.032381,-0.542784,-0.703656,0.0,-0.234203,2.0,-0.061694,-0.017511,0.004846,null,null,-0.412718,null,-0.602005,-0.307721,null,-0.264078,…,0.0,-0.054053,null,0.0,0.0,0.0,-0.290711,0.0,-0.061488,0.0,0.0,1.0,null,-0.210019,0.0,null,0.0,null,0.0,0.0,0.0,0.0,0.0,-0.00859,0.0,-0.046292,-0.097437,1.0,0.0,0.0,0.0,null,0.0,-0.040013,212.0,0.0,0.0
1000004,null,null,null,null,null,null,null,null,null,null,null,null,-0.475778,-0.070289,-0.104497,-0.983056,-0.57313,-0.591927,-0.341678,-0.02422,-0.542784,null,2.0,null,0.0,-0.063741,-0.013377,null,-0.005735,null,0.862448,null,-0.724265,-0.307721,null,0.688482,…,2.0,-0.054053,null,2.0,0.0,2.0,-0.290711,2.0,-0.061488,2.0,1.0,2.0,null,null,2.0,null,0.0,null,0.0,2.0,2.0,0.0,2.0,-0.00859,0.0,-0.046292,-0.097437,1.0,1.0,2.0,2.0,null,2.0,-0.040013,212.0,2.0,2.0
1000005,-0.004519,null,-0.019913,-0.018674,-0.025267,-0.027545,null,null,null,null,null,null,null,0.553573,-0.155999,null,-0.57313,-0.591927,-0.341678,0.060004,null,null,0.0,0.276901,2.0,-0.051576,0.026633,-0.128137,null,null,-0.385586,null,null,-0.307721,-0.544196,-0.264078,…,1.0,-0.054053,null,0.0,2.0,0.0,-0.290711,0.0,null,0.0,0.0,0.0,null,-0.210019,0.0,null,2.0,null,2.0,0.0,0.0,2.0,0.0,null,2.0,null,-0.097437,1.0,2.0,0.0,0.0,null,0.0,null,212.0,0.0,0.0


In [10]:
test_main = test[['customer_id'] + top_main_features]
test_main.head()

customer_id,num_feature_41,num_feature_87,num_feature_27,num_feature_117,num_feature_62,num_feature_76,num_feature_42,num_feature_71,num_feature_25,num_feature_88,num_feature_22,num_feature_23,num_feature_126,num_feature_35,num_feature_85,num_feature_36,num_feature_116,num_feature_68,num_feature_73,num_feature_46,num_feature_24,num_feature_61,cat_feature_8,num_feature_95,cat_feature_66,num_feature_58,num_feature_60,num_feature_26,num_feature_16,num_feature_69,num_feature_15,num_feature_52,num_feature_130,num_feature_6,num_feature_63,num_feature_104,…,cat_feature_15,num_feature_86,num_feature_45,cat_feature_37,cat_feature_35,cat_feature_23,num_feature_65,cat_feature_42,num_feature_103,cat_feature_57,cat_feature_14,cat_feature_55,num_feature_118,num_feature_2,cat_feature_5,num_feature_74,cat_feature_33,num_feature_112,cat_feature_25,cat_feature_53,cat_feature_31,cat_feature_21,cat_feature_18,num_feature_93,cat_feature_65,num_feature_92,num_feature_78,cat_feature_4,cat_feature_48,cat_feature_19,cat_feature_54,num_feature_44,cat_feature_51,num_feature_37,cat_feature_34,cat_feature_30,cat_feature_61
i32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1750001,0.960263,null,-0.018283,-0.01556,-0.024379,-0.027352,null,null,null,null,null,null,-0.475778,-0.227345,null,-0.436894,-0.493959,0.036692,-0.341678,-0.023483,-0.542784,null,2.0,-0.121838,0.0,-0.019656,-0.012998,-0.139706,-0.005735,null,-0.602636,-0.066879,-0.765018,-0.307721,null,-0.383148,…,0.0,-0.054053,null,0.0,0.0,2.0,-0.290711,2.0,-0.061488,2.0,1.0,2.0,null,-0.210019,2.0,null,0.0,null,0.0,0.0,2.0,0.0,2.0,-0.00859,0.0,-0.046292,-0.097437,0.0,0.0,2.0,2.0,null,2.0,-0.040013,212.0,2.0,0.0
1750002,-0.712026,null,-0.016736,-0.015727,-0.024826,-0.026929,null,null,null,null,null,null,-0.475778,0.954938,null,1.080224,-0.57313,-0.363338,-0.341678,-0.018151,-0.514319,-0.346709,2.0,-0.13211,2.0,-0.038366,-0.01038,0.004846,null,null,-0.222799,null,-0.357487,-0.307721,null,-0.025938,…,0.0,-0.054053,null,0.0,1.0,2.0,-0.290711,2.0,0.367167,2.0,1.0,2.0,null,-0.210019,2.0,null,0.0,null,0.0,0.0,2.0,0.0,2.0,-0.00859,1.0,-0.046292,-0.097437,0.0,0.0,2.0,2.0,null,2.0,-0.040013,212.0,2.0,0.0
1750003,1.153219,null,-0.020352,-0.01934,-0.025318,-0.027116,null,null,null,null,null,null,-0.455446,null,null,-0.13347,null,null,null,-0.009617,-0.229675,null,2.0,-0.234203,0.0,-0.05632,-0.005885,null,-0.005735,null,-0.439849,-0.066879,-0.785394,null,-0.035788,-0.264078,…,2.0,null,null,0.0,0.0,2.0,null,2.0,-0.061488,2.0,1.0,2.0,null,null,2.0,null,0.0,null,0.0,0.0,2.0,0.0,2.0,-0.00859,0.0,null,null,1.0,0.0,2.0,2.0,null,2.0,null,212.0,2.0,2.0
1750004,0.381394,null,-0.018194,-0.015257,-0.025244,-0.027451,null,null,null,null,null,null,-0.475778,-0.521825,null,0.048584,-0.018932,0.265281,0.934265,-0.04039,-0.542784,null,2.0,null,2.0,-0.050907,-0.021672,null,null,null,-0.602636,null,-0.520499,-0.307721,null,-0.383148,…,2.0,-0.054053,null,2.0,0.0,2.0,2.991479,2.0,-0.061488,2.0,1.0,2.0,null,null,2.0,null,0.0,null,0.0,2.0,2.0,0.0,2.0,-0.00859,0.0,-0.046292,-0.097437,0.0,0.0,2.0,2.0,null,2.0,-0.040013,212.0,2.0,2.0
1750005,null,null,-0.017956,-0.017804,-0.024389,-0.027126,null,null,null,null,null,null,1.533778,0.322352,null,1.444332,-0.57313,-0.020455,-0.341678,-0.023725,-0.514319,null,2.0,-0.234006,2.0,-0.020805,-0.013123,null,null,null,-0.602636,null,0.314939,-0.307721,null,-0.383148,…,2.0,-0.054053,null,0.0,0.0,2.0,-0.290711,2.0,-0.061488,2.0,1.0,2.0,null,null,2.0,null,0.0,null,1.0,0.0,2.0,0.0,2.0,-0.00859,0.0,-0.046292,-0.097437,0.0,0.0,2.0,2.0,null,2.0,-0.040013,212.0,2.0,2.0


In [11]:
lazy_extra = pl.scan_parquet('/kaggle/input/datasets/mashamalyshkina/kyberpolka-dataset/train_extra_features.parquet')

lazy_extra = lazy_extra.cast(pl.Float64)

lazy_extra = lazy_extra.select(['customer_id'] + top_extra_features)

train_extra = lazy_extra.collect()

In [12]:
train_extra = train_extra.with_columns(
    [pl.col(c).cast(pl.Float32) for c in train_extra.columns if train_extra[c].dtype == pl.Float64]
)
train_extra = train_extra.with_columns(
    pl.col('customer_id').cast(pl.Int32)
)

In [13]:
test_extra = test_extra[['customer_id'] + top_extra_features]

In [14]:
test_extra = test_extra.with_columns(
    [pl.col(c).cast(pl.Float32) for c in train_extra.columns if test_extra[c].dtype == pl.Float64]
)
test_extra = test_extra.with_columns(
    pl.col('customer_id').cast(pl.Int32)
)

In [15]:
train_combined = train_main.join(
    train_extra,
    on='customer_id',
    how='left'  
)

print("Размер после объединения train и extra:", train_combined.height)
print("Количество столбцов:", train_combined.width)

train_combined.write_parquet('train_combined_1600.parquet')

Размер после объединения train и extra: 750000
Количество столбцов: 1624


In [16]:
test_combined = test_main.join(
    test_extra,
    on='customer_id',
    how='left'  
)

print("Размер после объединения test и extra:", test_combined.height)
print("Количество столбцов:", test_combined.width)

test_combined.write_parquet('test_combined_1600.parquet')

Размер после объединения test и extra: 250000
Количество столбцов: 1624
